In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# df_site = pd.read_excel(r'Ethiopia DTM - SA R38_Dataset_Sharable without GPS.xlsx', sheet_name='Sites')

In [ ]:
df_site = pd.read_excel(r'Ethiopia DTM - R39_Dataset_EC.xlsx')

In [ ]:
# df_site_south = pd.concat([df_site, df_south_et])

In [ ]:
# df_site= df_site_south

In [ ]:
# df_site.to_excel('Merged data frame with south Ethiopia.xlsx')

In [ ]:
df_site.columns.to_list()

In [ ]:
df_site['M-0342: Settlement/site type'].value_counts()

In [ ]:
df_site['Recoded_comp'] = df_site['M-0342: Settlement/site type'].replace({'Spontaneous camp/site': 'In Camp', 
                                                                'Collective center': 'In Camp', 
                                                                'Planned camp/site': 'In Camp',
                                                                'Host community': 'Out of Camp',
                                                                'Dispersed settlement':'Out of Camp'})

In [ ]:
df_site['Recoded_comp'].value_counts()

In [ ]:
df_zonal_level = df_site.groupby(['M-0303: Region','OCHA Zone'])['M-0309: Total Number of IDP HHs'].sum().reset_index()

In [ ]:
zonal_level = df_site.groupby(['M-0303: Region','OCHA Zone','Recoded_comp'], dropna=False)['M-0309: Total Number of IDP HHs'].sum().reset_index()

In [ ]:
df_zonal_level['M-0309: Total Number of IDP HHs'].sum()

In [ ]:
zonal_level 

In [ ]:
df_zonal_cross =  pd.crosstab(index= [zonal_level['M-0303: Region'], zonal_level['OCHA Zone']],
                                 columns= zonal_level['Recoded_comp'],
                                     values=zonal_level['M-0309: Total Number of IDP HHs'],
                                 aggfunc='sum'
                                        ).reset_index()

In [ ]:
df_zonal_cross

In [ ]:
df_woreda_cross =  pd.crosstab(index= [df_site['M-0303: Region'], df_site['OCHA Zone'], df_site['OCHA Woreda']],
                                 columns= df_site['Recoded_comp'],
                                 values = df_site['M-0309: Total Number of IDP HHs'],
                                 aggfunc='sum'
                                        ).reset_index()

In [ ]:
# Zo_exclusion = df_zonal_cross[(df_zonal_cross['In Camp'].notna()) & (df_zonal_cross['Out of Camp'].notna())]

In [ ]:
# Woreda_pop = df_woreda_cross[(df_woreda_cross['In Camp'].notna()) & (df_woreda_cross['Out of Camp'].notna())]

In [ ]:
# Woreda_pop.shape

In [ ]:
# Zo_exclusion.columns

In [ ]:
# df_woreda_cross['Out of Camp'].sum()

In [ ]:
def calculate_sample_size(df, population_size_col):
    
    Z = 1.645  # Z-score for 90% confidence level
    P = 0.5   # Proportion (assumed maximum variability)
    d = 0.1  # Margin of error
    deff = 1.5  # Design Effect

    def sample_size_formula(N):
        
        numerator = (Z**2) * P * N * (1 - P) * deff
        denominator = (d**2) * (N - 1) + (Z**2) * P * (1 - P)

        
        if denominator == 0:
            return np.nan
        
        
        sample_size = np.ceil(numerator / denominator)
        return min(sample_size, N)  

   
    df['Sample_Size_incomp'] = df['In Camp'].apply(sample_size_formula)
    df['Sample_Size_outofcamp'] = df['Out of Camp'].apply(sample_size_formula)
    
    return df



df = pd.DataFrame(df_zonal_cross)


df = calculate_sample_size(df, 'Population_Size')


In [ ]:
df['_reserve_samples_incomp'] = ((df['Sample_Size_incomp']*20)/100).round(0)

In [ ]:
df['_reserve_samples_Outofcomp'] = ((df['Sample_Size_outofcamp']*20)/100).round(0)

In [ ]:
df['In ccmp Sample size'] = df[['_reserve_samples_incomp', 'Sample_Size_incomp']].sum(axis=1)

In [ ]:
df['Out of ccmp Sample size'] = df[['_reserve_samples_Outofcomp', 'Sample_Size_outofcamp']].sum(axis=1)

In [ ]:
df['Out of ccmp Sample size'].sum()

In [ ]:
df['In ccmp Sample size'].sum()

In [ ]:
7402+5761    ##samples shared with Boss =13725

In [ ]:
# df.to_excel('dataframe_sample.xlsx')

In [ ]:
# df['Number of clusters to be selected'] = df['In ccmp Sample size'] / 17

In [ ]:
df_sample_incomp = df[['M-0303: Region', 'OCHA Zone','In ccmp Sample size','In Camp']] 

In [ ]:
df_sample_incomp['Number of clusters to be selected'] = (df_sample_incomp['In ccmp Sample size'] / 17).round(0)

In [ ]:
df_sample_ouofcomp = df[['M-0303: Region', 'OCHA Zone','Out of ccmp Sample size','Out of Camp']] 

In [ ]:
df_sample_ouofcomp['Number of clusters to be selected'] = (df_sample_ouofcomp['Out of ccmp Sample size'] / 17).round(0)

In [ ]:
df_sample_ouofcomp['Number of clusters to be selected'].describe()

In [ ]:
df_site2 = df_site[['M-0303: Region','OCHA Zone','OCHA Woreda', 'M-0445: Site ID', 'M-0448: Site Name','M-0309: Total Number of IDP HHs', 'Recoded_comp']]

In [ ]:
df_incomp_sites = df_site2[df_site2['Recoded_comp'] == 'In Camp']
df_outofcomp_sites = df_site2[df_site2['Recoded_comp'] == 'Out of Camp']

#### Sampling At Site level 

### In Camp

In [ ]:
# In-camp sampling 

# Initialize output columns
df_incomp_sites.loc[:, 'Site Status of Selection'] = 'Not selected'
df_incomp_sites.loc[:, 'Sample to be Interviewed per Selected Site'] = 0
df_incomp_sites.loc[:, 'clusters and sample summary'] = ''

# Sort by zone and descending population
df_incomp_sites = df_incomp_sites.sort_values(
    ['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False]
)

# Process each OCHA Zone
for _, row in df_sample_incomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['In ccmp Sample size'])

    # Filter the sites for this zone
    zone_sites = df_incomp_sites[df_incomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine number of sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top-N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark selected
    df_incomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Sum of available population in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Use actual population if sample size is too high
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Integer division and remainder to ensure no rounding issues
        base_sample = final_sample_size // n_to_select
        remainder = final_sample_size % n_to_select

        # Assign base sample to all selected sites
        df_incomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = base_sample

        # Distribute the remainder to the first 'remainder' sites
        if remainder > 0:
            extra_indices = selected_indices[:remainder]
            df_incomp_sites.loc[extra_indices, 'Sample to be Interviewed per Selected Site'] += 1

        # Create and assign summary string to all rows in the zone
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_incomp_sites.loc[df_incomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# Optional preview
# print(df_incomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                        'Site Status of Selection', 'Sample to be Interviewed per Selected Site',
#                        'clusters and sample summary']])


### Out of camp 

In [ ]:
# Initialize output columns safely
df_outofcomp_sites.loc[:, 'Site Status of Selection'] = 'Not selected'
df_outofcomp_sites.loc[:, 'Sample to be Interviewed per Selected Site'] = 0
df_outofcomp_sites.loc[:, 'clusters and sample summary'] = ''

# Sort by zone and descending population
df_outofcomp_sites = df_outofcomp_sites.sort_values(
    ['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False]
)

# Process each OCHA Zone
for _, row in df_sample_ouofcomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['Out of ccmp Sample size'])

    # Filter the sites for this zone
    zone_sites = df_outofcomp_sites[df_outofcomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine number of sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top-N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark selected
    df_outofcomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Sum of available population in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Use actual population if sample size is too high
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Integer division and remainder to avoid rounding differences
        base_sample = final_sample_size // n_to_select
        remainder = final_sample_size % n_to_select

        # Assign base sample to all selected sites
        df_outofcomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = base_sample

        # Distribute remainder to the first 'remainder' sites
        if remainder > 0:
            extra_indices = selected_indices[:remainder]
            df_outofcomp_sites.loc[extra_indices, 'Sample to be Interviewed per Selected Site'] += 1

        # Create and assign summary string to all rows in the zone
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_outofcomp_sites.loc[df_outofcomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# Optional preview
# print(df_outofcomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                           'Site Status of Selection', 'Sample to be Interviewed per Selected Site',
#                           'clusters and sample summary']])


In [ ]:
with pd.ExcelWriter('Sampled_sites_Incomp and Out of Camp_r39.xlsx', engine='xlsxwriter') as writer:
    df_incomp_sites.to_excel(writer, sheet_name='In-Camp Sites', index=False)
    df_outofcomp_sites.to_excel(writer, sheet_name='Out-of-Camp Sites', index=False)


In [ ]:
# Initialize output columns
df_incomp_sites['Site Status of Selection'] = 'Not selected'
df_incomp_sites['Sample to be Interviewed per Selected Site'] = 0
df_incomp_sites['clusters and sample summary'] = ''

# Sort sites by zone and descending IDP HHs
df_incomp_sites = df_incomp_sites.sort_values(['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False])

# Process each OCHA Zone
for _, row in df_sample_incomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['In ccmp Sample size'])

    # Filter the sites for this zone
    zone_sites = df_incomp_sites[df_incomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine number of sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark selected
    df_incomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Sum of available population in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Use actual population if sample size is too high
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Calculate sample per site (rounded)
        sample_per_site = round(final_sample_size / n_to_select)

        # Assign to selected sites
        df_incomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = sample_per_site

        # Create summary string
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_incomp_sites.loc[df_incomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# Preview result
# print(df_incomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
                       # 'sampled site', 'sample size per site', 'clusters and sample summary']])


In [ ]:
# Initialize output columns
df_outofcomp_sites['Site Status of Selection'] = 'Not selected'
df_outofcomp_sites['Sample to be Interviewed per Selected Site'] = 0
df_outofcomp_sites['clusters and sample summary'] = ''

# Sort by size within zone
df_outofcomp_sites = df_outofcomp_sites.sort_values(
    ['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False]
)

# Loop through each zone in sampling plan
for _, row in df_sample_ouofcomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['Out of ccmp Sample size'])

    # Filter all sites in this zone
    zone_sites = df_outofcomp_sites[df_outofcomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine how many sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top-N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark them as selected
    df_outofcomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Calculate the total IDP HHs in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Adjust sample size if needed
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Calculate equal share per site (rounded to 0 decimals)
        sample_per_site = round(final_sample_size / n_to_select)

        # Assign sample size per site
        df_outofcomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = sample_per_site

        # Assign summary string to all sites in this zone
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_outofcomp_sites.loc[df_outofcomp_sites['OCHA Zone'] == zone, 'Clusters number per zone and Sample summary'] = summary_text

# Optional: Preview result
# print(df_outofcomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                           'sampled site', 'sample size per site', 'clusters and sample summary']])


In [ ]:

# Initialize output columns
df_outofcomp_sites['sampled site'] = 'not selected'
df_outofcomp_sites['sample size per site'] = 0
df_outofcomp_sites['clusters and sample summary'] = ''

# Sort by size within zone
df_outofcomp_sites = df_outofcomp_sites.sort_values(['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False])

# Loop through each zone
for _, row in df_sample_ouofcomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    total_sample_size = int(row['Out of ccmp Sample size'])

    # Get all sites in the zone
    zone_sites = df_outofcomp_sites[df_outofcomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Decide how many to select
    n_to_select = min(n_clusters, available_sites)

    # Get indices of selected sites
    selected_indices = zone_sites.head(n_to_select).index

    # Mark as selected
    df_outofcomp_sites.loc[selected_indices, 'sampled site'] = 'selected'

    # Calculate and assign sample size per site (rounded)
    if n_to_select > 0:
        sample_per_site = round(total_sample_size / n_to_select)
        df_outofcomp_sites.loc[selected_indices, 'sample size per site'] = sample_per_site

        # Create summary string
        summary_text = f'Clusters: {n_to_select}, Sample: {total_sample_size}'
        df_outofcomp_sites.loc[df_outofcomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# # Preview result
# print(df_outofcomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                        'sampled site', 'sample size per site', 'clusters and sample summary']])

In [ ]:
# Initialize output columns
df_incomp_sites['sampled site'] = 'not selected'
df_incomp_sites['sample size per site'] = 0
df_incomp_sites['clusters and sample summary'] = ''

# Sort by size within zone
df_incomp_sites = df_incomp_sites.sort_values(['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False])

# Loop through each zone
for _, row in df_sample_incomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    total_sample_size = int(row['In ccmp Sample size'])

    # Get all sites in the zone
    zone_sites = df_incomp_sites[df_incomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Decide how many to select
    n_to_select = min(n_clusters, available_sites)

    # Get indices of selected sites
    selected_indices = zone_sites.head(n_to_select).index

    # Mark as selected
    df_incomp_sites.loc[selected_indices, 'sampled site'] = 'selected'

    # Calculate and assign sample size per site (rounded)
    if n_to_select > 0:
        sample_per_site = round(total_sample_size / n_to_select)
        df_incomp_sites.loc[selected_indices, 'sample size per site'] = sample_per_site

        # Create summary string
        summary_text = f'Clusters: {n_to_select}, Sample: {total_sample_size}'
        df_incomp_sites.loc[df_incomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# # Preview result
# print(df_incomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                        'sampled site', 'sample size per site', 'clusters and sample summary']])


In [ ]:
df_sample_melted =  df.melt(id_vars=['M-0303: Region', 'OCHA Zone'], 
                                        value_vars=['Sample_Size_incomp', 'Sample_Size_outofcamp'],
                                       value_name='Sample Size',
                                       var_name='Pop_group')
# df_sample_melted = df_sample_melted[df_sample_melted['Sample Size'] != 0]

In [ ]:
df_sample_melted 

In [ ]:
df_sample_melted['camp type'] = df_sample_melted['Pop_group'].replace({'Out of ccmp Sample size': 'Out of Camp',
                                                          'In ccmp Sample size':'In Camp'}) 

In [ ]:
df_sample_melted

In [ ]:
df_site2 = df_site[['M-0303: Region','OCHA Zone','OCHA Woreda', 'M-0445: Site ID', 'M-0448: Site Name','M-0309: Total Number of IDP HHs', 'Recoded_comp']]

In [ ]:
df_site3 = pd.crosstab(index= [df_site2['M-0303: Region'],df_site2['OCHA Zone']], 
                       columns=df_site2['Recoded_comp'], 
                       values=df_site2['M-0445: Site ID'],
                      aggfunc='count').reset_index()

In [ ]:
df_site3.columns

In [ ]:
df_site_final_mel=  df_site3.melt(id_vars=['M-0303: Region', 'OCHA Zone'], 
                                        value_vars=['In Camp', 'Out of Camp'],
                                       var_name='camp type',
                                       value_name='Total number of IDP camps')

In [ ]:
df_site2

In [ ]:
df_site_merged = df_sample_melted.merge(df_site_final_mel, on=['M-0303: Region','OCHA Zone','camp type'], how='inner' )

In [ ]:
# df_site_merged.to_excel(r'Main Final 150 samples.xlsx')

In [ ]:
df_site2

In [ ]:
df_merged_pps_site2 = df_sa_pps.merge(df_site2[['M-0445: Site ID', 'Recoded_comp','M-0309: Total Number of IDP HHs']], on='M-0445: Site ID', how='inner' )

In [ ]:
df_merged_pps_site2

In [ ]:
df_site4 = pd.crosstab(index= [df_merged_pps_site2['M-0303: Region'],df_merged_pps_site2['OCHA Zone']], 
                       columns=df_merged_pps_site2['Recoded_comp'], 
                       values=df_merged_pps_site2['M-0445: Site ID'],
                      aggfunc='count').reset_index()

In [ ]:
df_site3

In [ ]:
df_sample_melted.columns

In [ ]:
df_sam = pd.crosstab(index= [df_sample_melted['M-0303: Region'],df_sample_melted['OCHA Zone']], 
                           columns=df_sample_melted['Pop_group'], 
                       values=df_sample_melted['Sample Size'],
                      aggfunc='sum').reset_index()

In [ ]:
df_sam 

In [ ]:
df_mer = df_sam.merge(df_site4, on=['M-0303: Region', 'OCHA Zone'], how='inner' )

In [ ]:
# df_mer.to_excel('New sample and clusters merged.xlsx')

In [ ]:
df_mer

In [ ]:
# df['Total sample size reserves']= (df['sample']*20)/100

In [ ]:
# df['Sample_Size_outofcamp'].sum()

In [ ]:
# df['Total Sample size'] = df[['Sample_Size_incomp','Total sample size reserves']].sum(axis=1)

### Determining the number of Clusters

In [ ]:
# df.to_excel('Final _DE_1.5_nei.xlsx')

In [ ]:
df_sample_size.columns

In [ ]:
df_melted_group =  df_sample_melted.groupby(['M-0303: Region', 'OCHA Zone','Pop_group']) ['Sample Size'].sum().reset_index()

In [ ]:
# df['reserve']=  df['Sample_Size_outofcamp'] 

In [ ]:
df_melted_group

In [ ]:
# df_melted_group.to_excel('Melted_grouped.xlsx')


In [ ]:
df_site2 = df_site[['M-0303: Region','OCHA Zone','OCHA Woreda','Recoded_comp', 'M-0445: Site ID', 'M-0448: Site Name','M-0309: Total Number of IDP HHs', 'Recoded_comp']]

In [ ]:
df_site2 

In [ ]:

df_site2['pps_rank'] = df_site2.groupby('OCHA Zone')['M-0309: Total Number of IDP HHs'].rank(ascending=True, method='first').astype(int)


In [ ]:
# df_site.to_excel('final for PPs based on zonal.xlsx')

In [ ]:
df_site2['percentage_of_HHs'] = df_site2.groupby('OCHA Zone')['M-0309: Total Number of IDP HHs'].apply(
    lambda x: (x / x.sum() * 100).round(2)).reset_index(level=0, drop=True)

In [ ]:
# df_site2.to_excel('final for PPs based on zonal_pps.xlsx')

In [ ]:
df_site2

In [ ]:
# df_site2.to_excel('Cluster_size_final for PPs based on zonal_pps.xlsx')

#### Determining the number of Clusters 

In [ ]:
# df_site_cluster =  df_site2.groupby(['M-0303: Region', 'OCHA Zone','Recoded_comp']) ['M-0445: Site ID'].count().reset_index()

In [ ]:
df_site_cluster

In [ ]:
df_site_cluster.drop_duplicates(keep='first')

In [ ]:
df_final = df_site_cluster.merge(df_melted_group, on=['M-0303: Region', 'OCHA Zone'], how='right')

In [ ]:
df_final['Sample Size'].sum()

In [ ]:
df_44.shape

In [ ]:
df_final.to_excel('Main for checks.xlsx')